# Reference-qualified and optimized meshes

This study compares meshes against a numerical, spatially and temporally qualified reference. The native CFL time step is retained for every trial; exact geometry is unchanged and each incidence is a separately rotated object case. Solver angles are stored in the solver frame, so object-frame custom curves use `+ incidence_deg` when converting them back.

Reference grids use strict conformal validation, exact geometry corner anchors and local edge-alignment repairs. Optimization uses strict conformal validation with `feature_anchors=False`, preserving the exact input geometry while validating all topology and enlarged-cell requirements. No feasible candidate found is a search outcome, not proof that the cell budget is impossible.


The compact layout reserves **5 cells from the final PEC bounding box to TFSF**, **4 from TFSF to the integration contour**, and **6 from the contour to PML**, on each side. PML keeps 12 cells per side. Geometry and phase origin translate together when fitting the domain.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import fdtdmesh as fd
from fdtdmesh.benchmarks import (
    SHAPES, make_simulation, ReferenceSettings, qualify_reference,
    optimize_mesh, Optimization,
)

RUN_CASE = True
MAX_EVALUATIONS = 200
OPTIMIZATION_SECONDS = 600
SELECTED_SHAPE = 'sun'
INCIDENCE_DEG = 160.0
FREQUENCY = 1e9  # Hz
SCALE_FACTOR = 1.2  # Dimensionless multiplier of the default 0.6-wavelength recipe scale
CASE_NAME = f"compact_5_4_6_{SELECTED_SHAPE}_{INCIDENCE_DEG:g}_factor{SCALE_FACTOR:g}_f{FREQUENCY:g}"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
STUDY_ROOT = PROJECT_ROOT / 'artifacts/notebooks/mesh_study'
REFERENCE_DIR = STUDY_ROOT / CASE_NAME / 'reference'
CELLS = [(64, 64), (96, 96), (128, 128)]  # Total axis cell budgets, including PML.
# A single tuple, e.g. (80, 80), is also accepted.
if isinstance(CELLS, tuple) and len(CELLS) == 2 and all(np.isscalar(v) for v in CELLS):
    CELL_BUDGETS = [CELLS]
else:
    CELL_BUDGETS = list(CELLS)
if not CELL_BUDGETS or any(
    not isinstance(b, (tuple, list)) or len(b) != 2 or
    any(isinstance(v, (bool, np.bool_)) or not isinstance(v, (int, np.integer)) or v <= 0 for v in b)
    for b in CELL_BUDGETS
):
    raise ValueError("CELLS must be a nonempty list of positive integer (Nx, Ny) pairs, or one pair")
CELL_BUDGETS = list(dict.fromkeys(tuple(map(int, b)) for b in CELL_BUDGETS))
OPTIMIZATION_DIR = STUDY_ROOT / CASE_NAME / 'optimization'  # Exact JSON settings select a subdirectory automatically.
REF_SETTINGS = ReferenceSettings(ppw=(48, 72, 108, 164, 196, 256), rtol=.005, worst_rtol=.005, max_seconds=900)

# Large mesh figures for inspecting grid lines and conformal cut cells.
MESH_FIGSIZE = (14, 12)

def large_mesh(fig):
    fig.set_size_inches(*MESH_FIGSIZE, forward=True)
    fig.set_dpi(120)
    return fig


Set **SCALE_FACTOR** above to resize the exact shapes (1.5 means 50% larger than the default). The catalog preview, selected solve and best-mesh inspection share this setting. The lower-level `scale=` argument remains SI metres per recipe unit; it is not a multiplier. Output directories include size and frequency to separate reference caches. The optional `run_sweep` below uses its own default size and frequency.

## Procedural shape catalog

All thumbnails use `make_simulation` and `Simulation.plot_geometry` before meshing. The view expands with SCALE_FACTOR around the domain centre so enlarged features remain visible.

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(8, 21), dpi=100)
for ax, shape in zip(axes.flat, SHAPES):
    thumb = make_simulation(shape=shape, incidence_deg=0, frequency=FREQUENCY, scale_factor=SCALE_FACTOR)
    thumb.plot_geometry(mesh=False, units='wavelength', ax=ax)
    lam = thumb.wavelength
    cx, cy = np.asarray(thumb.size) / 2 / lam
    ax.set_xlim(cx - max(1, SCALE_FACTOR), cx + max(1, SCALE_FACTOR)); ax.set_ylim(cy - max(1, SCALE_FACTOR), cy + max(1, SCALE_FACTOR))
    ax.set_title(shape); ax.get_legend().remove()
for ax in axes.flat[len(SHAPES):]: ax.axis('off')
fig.suptitle(f'Exact geometry catalog · scale factor {SCALE_FACTOR:g}')
fig.tight_layout()
plt.show()

## One reference, multiple optimization budgets

Set `CELLS` to a list of `(Nx, Ny)` tuples above. Every budget shares the same qualified reference and gets its own optimization archive selected by the complete JSON settings. Evaluation and time limits apply **per budget**. Each budget gets a large best-mesh figure; unsuccessful searches get an explicit status panel. A single tuple remains supported.

Reference qualification checks real spatial and temporal differences, PML sensitivity, and contour sensitivity. An unqualified reference is displayed and all optimization runs are skipped.


In [ ]:
def _progress(item):
    status = item.get('status', 'running')
    level = item.get('ppw', item.get('number', ''))
    difference = item.get("difference") or {}
    error = item.get("error", item.get("best_error", difference.get("error", "")))
    print(f"{status:>22}  {level}  error={error}")
    if item.get("message"):
        print("  ", item["message"])

sim = make_simulation(shape=SELECTED_SHAPE, incidence_deg=INCIDENCE_DEG, frequency=FREQUENCY, scale_factor=SCALE_FACTOR)
reference = None
optimizations = {}
if RUN_CASE:
    reference = qualify_reference(sim, directory=REFERENCE_DIR, settings=REF_SETTINGS, progress=_progress)
    print('Reference:', reference.report.get('status'), 'qualified=', reference.qualified)
    print('Reference description:', reference.directory / 'experiment.json')
    if reference.result is not None:
        display(large_mesh(reference.result.plot_geometry(mesh=True, units='wavelength')))
        plt.close(plt.gcf())
    if reference.qualified:
        for budget in CELL_BUDGETS:
            print(f'\nCell budget: {budget[0]} × {budget[1]}')
            optimization = optimize_mesh(
                sim, reference, cells=budget, strategy='differential_evolution', feature_anchors=False,
                directory=OPTIMIZATION_DIR, max_evaluations=MAX_EVALUATIONS, max_seconds=OPTIMIZATION_SECONDS,
                controls=6, population=12, seed=0, progress=_progress,
            )
            print('Optimization:', optimization.report.get('status'),
                  'best_error=', optimization.report.get('best_error'))
            print('Optimization description:', optimization.directory / 'experiment.json')
            optimizations[budget] = optimization
    else:
        print('Skipping optimization because the numerical reference is not qualified.')

In [ ]:
if reference is not None:
    levels = [x for x in reference.report.get('levels', []) if x.get('difference')]
    fig, ax = plt.subplots(figsize=(7, 4))
    if levels:
        ax.loglog([x['ppw'] for x in levels], [x['difference']['error'] for x in levels], 'o-', label='reference levels')
    checks = reference.report.get('checks', {})
    for label, value in checks.items():
        if 'error' in value: ax.axhline(value['error'], ls='--', label=label)
    ax.axhline(REF_SETTINGS.rtol, color='k', alpha=.35, label='rtol')
    ax.set(xlabel='points per wavelength', ylabel='complex relative L2 difference', title='Reference convergence and sensitivity')
    ax.legend(); ax.grid(True, which='both', alpha=.25)
    plt.show()
    display(reference.report)

In [ ]:
def _result_for_trial(opt, trial):
    return fd.Result.load(opt.directory / trial['result_file'])

for budget, optimization in optimizations.items():
    print(f'\nBudget {budget[0]} × {budget[1]}')
    baselines = {t['kind']: t for t in optimization.report['trials'] if t.get('kind') in ('uniform', 'deterministic')}
    rows = []
    for label, trial in baselines.items():
        if trial.get('result_file'):
            result = _result_for_trial(optimization, trial)
            rows.append((label, trial.get('error'), trial.get('gpu_ms'), result.diagnostics.get('dt'), trial.get('status'),
                         ))
    rows.append(('best', optimization.report.get('best_error'),
                 optimization.best.diagnostics.get('gpu_ms') if optimization.best else None,
                 optimization.best.diagnostics.get('dt') if optimization.best else None,
                 optimization.report.get('status'),
                 ))
    from collections import Counter
    print('Trial statuses:', dict(Counter(t['status'] for t in optimization.report['trials'])))
    print('available baseline / optimized trials')
    print('label              error          gpu_ms       dt             status')
    for row in rows: print(f'{row[0]:<18} {str(row[1]):<14} {str(row[2]):<12} {str(row[3]):<14} {row[4]}')
    fig = optimization.plot_history()
    fig.axes[0].set_title(f"Budget {budget[0]} × {budget[1]} · best feasible error")
    display(fig)
    plt.close(fig)

In [ ]:
for budget, optimization in optimizations.items():
    label = f'{budget[0]} × {budget[1]}'
    if optimization.best is None:
        fig, ax = plt.subplots(figsize=MESH_FIGSIZE, dpi=120)
        sim.plot_geometry(mesh=False, units='wavelength', ax=ax)
        ax.set_title(f'{label} · no feasible mesh found')
        ax.text(.5, .02, f"Status: {optimization.report.get('status')} · {len(optimization.report['trials'])} trials",
                transform=ax.transAxes, ha='center', bbox=dict(facecolor='white', alpha=.9))
        plt.show()
        continue
    print(f'\nBudget {label}')
    fig, ax = plt.subplots(figsize=MESH_FIGSIZE, dpi=120)
    optimization.best.plot_geometry(mesh=True, units='wavelength', ax=ax)
    cx, cy = np.asarray(sim.size) / (2 * sim.wavelength)
    ax.set(xlim=(cx-max(1, SCALE_FACTOR), cx+max(1, SCALE_FACTOR)), ylim=(cy-max(1, SCALE_FACTOR), cy+max(1, SCALE_FACTOR)), title=f'{label} · best feasible mesh · error={optimization.report["best_error"]:.3g}')
    plt.show()
    best_sim = make_simulation(shape=SELECTED_SHAPE, incidence_deg=INCIDENCE_DEG, frequency=FREQUENCY, scale_factor=SCALE_FACTOR)
    best_sim.apply_mesh(optimization.best.mesh)
    fig, ax = plt.subplots(figsize=MESH_FIGSIZE, dpi=120)
    best_sim.plot_discretization(units='wavelength', ax=ax)
    ax.set(xlim=(cx-max(1, SCALE_FACTOR), cx+max(1, SCALE_FACTOR)), ylim=(cy-max(1, SCALE_FACTOR), cy+max(1, SCALE_FACTOR)))
    ax.set_title(f'{label} · conformal discretization')
    plt.show()
    pairs = [(t['kind'], t.get('error')) for t in optimization.report['trials'] if t.get('kind') in ('uniform', 'deterministic')]
    print('baseline errors:', pairs)

    # Compare in the original object frame; all angle-dependent plots are polar.
    ref_result = reference.result
    index = len(ref_result.frequencies) // 2
    theta = ref_result.angles + np.deg2rad(INCIDENCE_DEG)
    order = np.argsort(np.mod(theta, 2*np.pi))
    theta = np.mod(theta[order], 2*np.pi)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={'projection': 'polar'})
    fig.suptitle(f'Budget {label}')
    for curve_label, result in [('reference', ref_result), ('best', optimization.best)]:
        values = (abs(result.far_field[index]), np.angle(result.far_field[index]), result.width[index])
        for ax, value in zip(axes, values):
            ax.plot(np.r_[theta, theta[0]+2*np.pi], np.r_[value[order], value[order][0]], label=curve_label)
    axes[1].set_ylim(-np.pi, np.pi)
    axes[1].set_rorigin(-np.pi)  # Signed phase is a radial level, not an angle rotation.
    for ax, title in zip(axes, ['Far-field magnitude', 'Far-field phase [rad]', 'Scattering width [m]']):
        ax.set_title(title)
        ax.legend(loc='upper left', bbox_to_anchor=(.9, 1.12))
    plt.show()
    print('Winner tighter-stop validation:', optimization.report.get('validation'))
    uniform = next((t.get('error') for t in optimization.report['trials'] if t['kind']=='uniform'), None)
    observed = reference.report.get('observed_reference_difference')
    if uniform is not None and observed is not None:
        gain = uniform - optimization.report['best_error']
        print(f'Absolute error reduction: {gain:.4g}; observed reference difference: {observed:.4g}')
        if gain <= observed:
            print('The improvement is not resolved beyond the observed reference difference.')


## Optional sweep

The sweep is disabled by default. It preserves per-incidence reference and optimization archives, including DE population/RNG checkpoints and Powell warm restarts. Rows retain unqualified cases for later inspection; they do not claim every shape is feasible or globally optimal.

In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    from fdtdmesh.benchmarks import run_sweep, plot_gallery
    sweep_rows = run_sweep(
        SHAPES, (0, 30, 60, 90), directory=STUDY_ROOT / 'sweep_compact_5_4_6', cells=(192, 192),
        reference_settings=REF_SETTINGS, strategy='differential_evolution', feature_anchors=False,
        max_evaluations=60, max_seconds=600, progress=_progress,
    )
    display(sweep_rows)

study_manifest = STUDY_ROOT / 'sweep_compact_5_4_6' / 'study.json'
if study_manifest.exists():
    from fdtdmesh.benchmarks import plot_gallery
    # One large figure per case avoids squeezing four dense meshes into one row.
    for shape in SHAPES:
        for incidence in (0, 30, 60, 90):
            fig = plot_gallery(STUDY_ROOT / 'sweep_compact_5_4_6', shapes=[shape], incidences=[incidence])
            display(large_mesh(fig))
            plt.close(fig)